Benjamin Arnold; 10/18/2026; Lab 7: Physics Engine

Part A-- Core Engine

A.1: Body

In [ ]:
import numpy as np

class Body:
    def __init__(self, m=1.0, r=(0.0, 0.0), v=(0.0, 0.0), radius=0.0,
                    fixed=False, name=""):
        self.m, self.radius = float(m), float(radius)
        self.fixed = bool(fixed)
        self.name = name
        
        self.r = np.array(r, dtype=float)
        self.v = np.array(v, dtype=float)
        self.f = np.zeros(2)

    @property
    def inv_m(self):
        return 0.0 if self.fixed else 1.0 / self.m

    def clear(self):
        self.f[:] = 0.0

    def add_force(self, F):
        self.f += np.asarray(F, dtype=float)

    def ke(self):
        return 0.5 * self.m * self.v.dot(self.v)

    def p(self):
        return self.m * self.v

    def __repr__(self):
        return f"Body({self.name}, m={self.m}, r={self.r}, v={self.v})"

A.2: Force

In [ ]:
class Force:
    def potential_energy(self, world):
        return 0.0

class Gravity(Force):
    def __init__(self, g=9.81):
        self.gmag = g
        self.g    = np.array([0.0, -g])

    def apply(self, world):
        for b in world.bodies:
            if not b.fixed:
                b.add_force(b.m * self.g)

    def potential_energy(self, world):
        return sum(b.m * self.gmag * b.r[1] for b in world.bodies if not b.fixed)

class Spring(Force):
    def __init__(self, i, j, k=1.0, L0=0.0):
        self.i, self.j, self.k, self.L0 = i, j, k, L0

    def length(self, world):
        a, b = world.bodies[self.i], world.bodies[self.j]
        d = b.r - a.r
        return a, b, d, np.linalg.norm(d)
    
    def apply(self, world):
        a, b, d, L = self.length(world)
        if L > 0:
            F = self.k * (L - self.L0) * (d / L)
            a.add_force(F)
            b.add_force(-F)

    def potential_energy(self, world):
        _, _, _, L = self.length(world)
        return 0.5 * self.k * (L - self.L0)**2

class Drag(Force):
    def __init__(self, k=0.1, n=1.0):
        self.k, self.n = k, n
        
    def apply(self, world):
        for b in world.bodies:
            if b.fixed:
                continue
            vmag = np.linalg.norm(b.v)
            if vmag == 0:
                continue
            b.add_force(-self.k*(vmag**self.n)*b.v)

A.3: Integrator

In [ ]:
class EulerCromer:
    def step(self, world, dt):
        world.compute_forces()
        for b in world.bodies:
            if b.fixed:
                continue
            b.v += (b.f*b.inv_m)*dt
            b.r += b.v*dt

A.4: World

In [ ]:
class World:
    def __init__(self, integrator=EulerCromer, forces=None, bodies=None):
        self.bodies      = list(bodies or [])
        self.forces      = list(forces or [])
        self.integrator  = integrator
        self.t           = 0.0
        self.recorders   = {}
        self.history     = {}

    def add(self, body):
        self.bodies.append(body); return body

    def add_force(self, force):
        self.forces.append(force); return force

    def compute_forces(self):
        for b in self.bodies:
            b.clear()
        for f in self.forces:
            f.apply(self)                  

    def step(self, dt):
        self.integrator.step(self, dt)   
        self.t += dt                        

    
    def kinetic_energy(self):
        return sum(b.ke() for b in self.bodies)
    def potential_energy(self):
        return sum(f.potential_energy(self) for f in self.forces)
    def total_energy(self):
        return self.kinetic_energy() + self.potential_energy()
    def momentum(self):
        return sum((b.p() for b in self.bodies), np.zeros(2))

    
    def record(self, name, fn):
        self.recorders[name] = fn
        self.history[name]   = []

    def capture(self):
        for k, fn in self.recorders.items():
            self.history[k].append(fn(self))

    def run(self, dt, t_end):
        self.capture()
        for _ in range(int(round(t_end / dt))):
            self.step(dt)
        return {k: np.array(v) for k, v in self.history.items()}

Part B-- Validation Against Old Labs

B.1: Lab 1-- Projectile Motion

In [ ]:
import matplotlib.pyplot as plt

world1=World(integrator=EulerCromer)
world1.add_force(Gravity)

theta, v0=np.pi/4, 40
proj=Body(v=[v0*np.cos(theta),v0*np.sin(theta)])
world1.add(proj)

world1.record('t', lambda W: W.t)
world1.record('x', lambda W: W.bodies[0].r[0])
world1.record('y', lambda W: W.bodies[0].r[1])
world1.record('E', lambda W: W.total_energy())

w=world1.run(dt=0.001,t_end=5)
x,y,E=w['x'],w['y'],w['E']

plt.figure(figsize=(8, 4))
plt.plot(x, y, linewidth=2, label='Parabolic Trajectory')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Projectile Trajectory')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

